# Run the study on Colab

**This notebook is the primary runner.** All 3DGS optimization happens here;
the Mac is for editing code, reading results, and building figures.

Run the cells in order, **including Fetch data**, every session — Colab wipes
`/content`, so the extract has to happen again.

## Google Drive is optional

`drive.mount()` commonly fails with a **400** from the VS Code extension, and
when several Google accounts are signed in. That is fine — cell 2 catches it
and carries on:

| | with Drive | without Drive |
|---|---|---|
| dataset archives | kept on Drive, downloaded once | re-downloaded each session (seconds) |
| extracted data | `/content` | `/content` |
| results | rsynced to Drive | **zipped for you to download by hand (cell 8)** |

Only the results actually matter, because they cost GPU-hours. The data does
not: HuggingFace to Colab is fast enough that re-downloading is cheaper than
fighting the auth.

### If you want Drive working anyway
In rough order of success rate:
1. Open this notebook in the **browser** at colab.research.google.com rather
   than the VS Code extension. The mount broker only exists in the Colab frontend.
2. Sign out of every Google account except the one you want, then retry.
3. `drive.mount('/content/drive', force_remount=True)`.

Storage layout when Drive does work: archives on Drive (FUSE is slow for
thousands of small PNGs, so never read extracted data from it), extracted data
and live runs on local disk, finished runs rsynced back.

In [1]:
#@title 1. What GPU did we get?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda)

/bin/bash: line 1: nvidia-smi: command not found
torch 2.11.0+cpu | cuda None


In [4]:
#@title 2. Paths (Drive optional)
# drive.mount() fails from the VS Code extension and when several Google
# accounts are signed in. It is only an optimization here: the archives
# re-download from HuggingFace in seconds on Colab. So try it, and carry on
# without it if it fails.
import os

DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    print('Drive mounted\n')
except Exception as e:
    print(f'Drive NOT mounted ({type(e).__name__}). Continuing without it.')
    print('Data still works -- it re-downloads each session, which is quick.')
    print('RESULTS will not persist; see cell 8.\n')

ARCHIVES   = f'{DRIVE}/datasets/blender' if DRIVE else '/content/archives'
RUNS_DRIVE = f'{DRIVE}/dl3dcv/runs'      if DRIVE else None
os.environ['DATA_ROOT'] = '/content/data'
os.environ['RUNS_ROOT'] = '/content/runs'

for d in (ARCHIVES, RUNS_DRIVE, '/content/runs'):
    if d: os.makedirs(d, exist_ok=True)

n = len([f for f in os.listdir(ARCHIVES) if f.endswith('.zip')])
print('archives  ', ARCHIVES, f'({n} present)',
      '[PERSISTENT]' if DRIVE else '[lost on disconnect - redownloads]')
print('DATA_ROOT ', os.environ['DATA_ROOT'], '(local, per-session)')
print('RUNS_ROOT ', os.environ['RUNS_ROOT'],
      '(synced to Drive)' if DRIVE else '(LOCAL ONLY - see cell 8)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted

archives   /content/drive/MyDrive/datasets/blender (0 present) [PERSISTENT]
DATA_ROOT  /content/data (local, per-session)
RUNS_ROOT  /content/runs (synced to Drive)


In [ ]:
#@title 3. Clone and install
REPO = 'https://github.com/AMEER7525/depth-prior-tax.git'  #@param {type:'string'}
%cd /content
![ -d final_project ] || git clone $REPO final_project
%cd /content/final_project
!git pull --ff-only || true
!bash scripts/setup_gpu.sh

## Fetch data

Downloads happen **here, not on your laptop** — Colab pulls from HuggingFace
at datacenter speed, straight into Drive. Downloading locally and syncing up
was measured at roughly 100 KB/s, which would take hours for the same data.

**Re-run this cell every session.** Colab wipes `/content`, so the extract has
to happen again — but the download is skipped once the archives are on Drive,
so later sessions take seconds rather than minutes.


In [ ]:
#@title 4. Download archives to Drive + extract to local disk
SCENES = 'lego chair ship'  #@param {type:'string'}
!python scripts/fetch_data.py --scenes $SCENES \
    --archives "$ARCHIVES" --extract "$DATA_ROOT"

In [ ]:
#@title 5. Sanity-check the loaders against real files
# src/data.py was written from the spec, not from the files. This is the
# first time it meets actual data -- check before spending GPU hours.
import sys; sys.path.insert(0, '/content/final_project')
import numpy as np
from src.data import load_blender_scene

SCENE = globals().get('SCENES', 'lego chair ship').split()[0]
sc = load_blender_scene(scene=SCENE, split='train')
v = sc.views[0]
print(f'{len(sc)} views | image {v.image.shape} {v.image.dtype} '
      f'range [{v.image.min():.2f}, {v.image.max():.2f}]')
print('K =\n', np.round(v.K, 2))
print('c2w =\n', np.round(v.c2w, 3))
print('camera distance from origin:', np.round(np.linalg.norm(v.c2w[:3, 3]), 3),
      '(NeRF-Synthetic cameras sit ~4.0 from the object)')
print('has GT depth:', sc.has_depth, '<- False is expected; Axis C needs it rendered')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(11, 4))
for a, i in zip(ax, range(3)):
    a.imshow(sc.views[i].image); a.set_title(sc.views[i].name); a.axis('off')
plt.tight_layout()

## Run

> **Cell 7 does not work yet.** `scripts/train.py` is still a stub, so every
> run fails immediately and writes a `FAILED` marker — 27 of them for stage1,
> and no `metrics.json`. Cells 1–6 are the useful part today: they set up the
> environment, fetch the data, and verify the loaders.

Once `train.py` is wired to gsplat: stages are sequential, each fixing the
previous one's winner. Edit `configs/sweep.yaml` between stages. `--resume`
skips finished runs, so re-running after a disconnect picks up where it
stopped.


In [ ]:
#@title 6. Restore finished runs (skips if no Drive)
import os, subprocess
if RUNS_DRIVE:
    subprocess.run(['rsync', '-a', RUNS_DRIVE + '/', os.environ['RUNS_ROOT'] + '/'])
    n = sum(1 for _, _, fs in os.walk(os.environ['RUNS_ROOT']) for f in fs
            if f == 'metrics.json')
    print(f'{n} finished run(s) restored from Drive')
else:
    print('No Drive: starting from whatever is in /content/runs (usually empty).')

In [ ]:
#@title 7. Launch a stage
STAGE = 'stage1'  #@param ['stage1','stage2','stage3_noise','stage3_affine','stage4_real']
!python scripts/run_sweep.py --sweep configs/sweep.yaml --stage $STAGE \
    --resume --explain-pruning

In [ ]:
#@title 8. Persist results (ALWAYS run before closing)
# Colab reclaims /content without warning. Anything unsaved is gone.
import os, shutil, subprocess
RUNS = os.environ['RUNS_ROOT']
n = sum(1 for _, _, fs in os.walk(RUNS) for f in fs if f == 'metrics.json')

if RUNS_DRIVE:
    subprocess.run(['rsync', '-a', RUNS + '/', RUNS_DRIVE + '/'])
    print(f'{n} run(s) synced to Drive')
else:
    # No Drive: metrics are small JSON, so a zip is enough to download by hand.
    shutil.make_archive('/content/results', 'zip', RUNS)
    mb = os.path.getsize('/content/results.zip') / 1e6
    print(f'{n} run(s) packed into /content/results.zip ({mb:.1f} MB)')
    print('DOWNLOAD IT NOW -- it disappears with the VM.')
    try:
        from google.colab import files; files.download('/content/results.zip')
    except Exception:
        print('(right-click the file in the Files pane to download)')

failed = [d for d, _, fs in os.walk(RUNS) if 'FAILED' in fs]
print(f'{len(failed)} failed run(s)')